# Sprint Review · Innovación Automotriz SpA

Revisión de estimaciones de sprint contra base histórica del equipo.

In [ ]:
# ============================================================
# CELDA 1 — Setup (no tocar después de crear)
# ============================================================
import pandas as pd
from IPython.display import display

EQUIPO = {
    "javelasquezb": {"nombre": "Javier Velásquez", "rol": "Tech Lead"},
    "Ecxpectro": {"nombre": "Henrique Schraiber", "rol": "Engineer"},
    "GuilhermeKill": {"nombre": "Guilherme Reis", "rol": "QA Engineer"},
}

# Base histórica por tipo de tarea: Hist / Meta Sprint / Desafío IA
BASE_HISTORICA = {
    "Configuración":               {"hist": 2.7, "meta": 2.1, "desafio": 1.5},
    "Corrección / Bug":             {"hist": 2.6, "meta": 2.0, "desafio": 1.4},
    "UI / Frontend":                {"hist": 3.8, "meta": 2.9, "desafio": 2.0},
    "Integración API":              {"hist": 3.7, "meta": 2.8, "desafio": 1.9},
    "Módulo / Lógica":              {"hist": 5.2, "meta": 3.8, "desafio": 2.5},
    "Notificaciones / Mensajería":  {"hist": 4.3, "meta": 3.3, "desafio": 2.3},
    "Reportes / Export":            {"hist": 6.1, "meta": 4.5, "desafio": 2.9},
    "Migración / Datos":            {"hist": 6.9, "meta": 5.1, "desafio": 3.3},
    "Feature General":              {"hist": 7.5, "meta": 5.4, "desafio": 3.5},
}

# Escala de puntos -> horas reales aproximadas
ESCALA_PUNTOS = {
    0.5: "~2h", 1: "~4h (½ día)", 2: "~8h (1 día)", 3: "~12h (1½ día)",
    5: "~20h (2½ días)", 8: "~32h (4 días)", 10: "~40h (5 días)",
}
# Regla: separar en sub-tareas si >= 8 pts (con apoyo IA) o >= 13 pts (sin IA)
UMBRAL_SEPARAR_CON_IA = 8
UMBRAL_SEPARAR_SIN_IA = 13

# Multiplicadores IA por talla (referencia)
MULTIPLICADORES_IA = {
    "S":  {"max_pts": 2,            "mult": 1.2},
    "M":  {"max_pts": 5,            "mult": 1.5},
    "L":  {"max_pts": 9,            "mult": 2.0},
    "XL": {"max_pts": float("inf"), "mult": 2.5},
}

# Reglas de clasificación de tipo por palabras clave — la primera coincidencia gana
REGLAS_TIPO = [
    ("Configuración", ["keyvault", "configur", "setup", "deploy", "azure", "system", "css", "batch", "worker"]),
    ("Integración API", ["integrac", "mercado libre", "api", "azure func"]),
    ("Módulo / Lógica", ["modulo", "módulo", "lógica", "logica", "calculo", "cálculo", "política", "politica", "proceso", "tooling", "chat ia", " ia ", "ia ", " ia", "systemprompt", "conversacion", "conversación"]),
    ("UI / Frontend", ["apartado", "mi cuenta", "cotizacion", "cotización", "radar", "listado", "busqueda", "búsqueda", "buscador", "buscar", "vista", "pantalla", "modal", "tabla"]),
    ("Corrección / Bug", ["revisar", "corregir", "arreglar", "revision", "revisión", "revison", "arreglo", "mejora"]),
    ("Migración / Datos", ["migrar", "migración", "migracion"]),
    ("Notificaciones / Mensajería", ["notificac", "whatsapp", "correo", "email", "recordatorio", "remarketing", "mensaje"]),
    ("Reportes / Export", ["descarga", "excel", "reporte", "export", "pdf"]),
]


def classify_task(titulo):
    t = f" {titulo.lower()} "
    for tipo, keywords in REGLAS_TIPO:
        if any(kw in t for kw in keywords):
            return tipo
    return "Feature General"


def clasificar_estado(est, base):
    if est <= base["desafio"]:
        return "DESAFÍO ✦"
    elif est <= base["meta"]:
        return "META ✓"
    elif est <= base["hist"]:
        return "OK"
    elif est <= base["hist"] * 1.3:
        return "ALTA ↑"
    else:
        return "INFLADA ↑↑"


def nota_estado(estado, est, base):
    mensajes = {
        "DESAFÍO ✦": f"Por debajo del desafío IA ({base['desafio']}) — excelente",
        "META ✓":    f"Dentro de la meta sprint ({base['meta']})",
        "OK":        f"Dentro del histórico ({base['hist']})",
        "ALTA ↑":    f"Hasta 30% sobre histórico ({base['hist']}) — revisar",
        "INFLADA ↑↑": f"Más de 30% sobre histórico ({base['hist']}) — revisar de cerca",
    }
    nota = mensajes[estado]
    if est >= UMBRAL_SEPARAR_SIN_IA:
        nota += f" · Considerar separar en sub-tareas (≥{UMBRAL_SEPARAR_SIN_IA} pts)"
    elif est >= UMBRAL_SEPARAR_CON_IA:
        nota += f" · Evaluar separar con apoyo IA (≥{UMBRAL_SEPARAR_CON_IA} pts)"
    return nota


def parse_tareas(tareas_str):
    filas = []
    for linea in tareas_str.strip().splitlines():
        linea = linea.strip()
        if not linea:
            continue
        if "|" not in linea:
            print(f"  ! línea ignorada (falta '|' con la estimación): {linea}")
            continue
        izq, est_str = linea.rsplit("|", 1)
        izq = izq.strip()
        if " - " in izq:
            id_, titulo = izq.split(" - ", 1)
        else:
            id_, titulo = "", izq
        try:
            est = float(est_str.strip())
        except ValueError:
            print(f"  ! línea ignorada (estimación inválida): {linea}")
            continue
        filas.append({"ID": id_.strip(), "Titulo": titulo.strip(), "Est": est})
    return filas


def analyze_sprint(nombre_sprint, tareas_str):
    filas = parse_tareas(tareas_str)
    if not filas:
        print("No se encontraron tareas válidas.")
        return

    registros = []
    for f in filas:
        tipo = classify_task(f["Titulo"])
        base = BASE_HISTORICA[tipo]
        estado = clasificar_estado(f["Est"], base)
        nota = nota_estado(estado, f["Est"], base)
        registros.append({
            "ID": f["ID"],
            "Título": f["Titulo"],
            "Est": f["Est"],
            "Tipo": tipo,
            "Hist": base["hist"],
            "Meta": base["meta"],
            "Desafío": base["desafio"],
            "Estado": estado,
            "Nota": nota,
        })

    df = pd.DataFrame(registros)

    print(f"{'='*100}\n{nombre_sprint} — Revisión de Estimaciones\n{'='*100}\n")
    display(df)

    total_tareas = len(df)
    pts_comprometidos = df["Est"].sum()
    pts_meta = df["Meta"].sum()
    pts_desafio = df["Desafío"].sum()

    print("\n📊 RESUMEN EJECUTIVO")
    print(f"  Total tareas:            {total_tareas}")
    print(f"  Puntos comprometidos:    {pts_comprometidos:.1f}")
    print(f"  Puntos si Meta Sprint:   {pts_meta:.1f}")
    print(f"  Puntos si Desafío IA:    {pts_desafio:.1f}")

    print("\n📈 DISTRIBUCIÓN DE ESTADOS")
    orden_estados = ["DESAFÍO ✦", "META ✓", "OK", "ALTA ↑", "INFLADA ↑↑"]
    conteo = df["Estado"].value_counts()
    for estado in orden_estados:
        n = int(conteo.get(estado, 0))
        barra = "█" * n
        print(f"  {estado:<12} {barra} ({n})")

    print("\n⚠️  TAREAS A REVISAR (ALTA / INFLADA)")
    revisar = df[df["Estado"].isin(["ALTA ↑", "INFLADA ↑↑"])]
    if revisar.empty:
        print("  Ninguna — todas las tareas están dentro de rango esperado.")
    else:
        for _, r in revisar.iterrows():
            print(f"  [{r['ID']}] {r['Título']} — {r['Estado']} ({r['Est']} pts) — {r['Nota']}")

    print("\n🏆 TAREAS EN DESAFÍO IA")
    desafio_tareas = df[df["Estado"] == "DESAFÍO ✦"]
    if desafio_tareas.empty:
        print("  Ninguna en este sprint.")
    else:
        for _, r in desafio_tareas.iterrows():
            print(f"  [{r['ID']}] {r['Título']} — {r['Est']} pts (vs desafío {r['Desafío']})")

In [ ]:
# ============================================================
# CELDA 2 — Datos del sprint (esta es la que se edita cada sprint)
# ============================================================
SPRINT = "Sprint 62"
TAREAS = """
196 - KeyVault | 2
197 - Generar politicas comerciales | 4
"""

In [ ]:
# ============================================================
# CELDA 3 — Ejecutar análisis
# ============================================================
analyze_sprint(SPRINT, TAREAS)